# Broadcast vs Shuffle vs Merge Joins

Small worked examples of the three Spark join strategies — **broadcast hash join**, **shuffle hash join**, and **sort-merge join** — on tiny sample DataFrames, with `explain()` output showing which physical operator Spark picks for each.

In [0]:
# Setup: a small customers dimension (small side) and an orders fact table (large side)
from pyspark.sql import functions as F

customers = spark.createDataFrame(
    [
        (1, "Alice",   "Hyderabad"),
        (2, "Bob",     "Bengaluru"),
        (3, "Charlie", "Chennai"),
        (4, "Diana",   "Pune"),
    ],
    schema="customer_id INT, name STRING, city STRING",
)

orders = spark.createDataFrame(
    [
        (101, 1, 250.00),
        (102, 2,  80.50),
        (103, 1, 120.75),
        (104, 3, 300.00),
        (105, 2,  45.25),
        (106, 4,  99.99),
    ],
    schema="order_id INT, customer_id INT, amount DOUBLE",
)

print("customers (small side):")
display(customers)
print("orders (large side):")
display(orders)

customers (small side):


customer_id,name,city
1,Alice,Hyderabad
2,Bob,Bengaluru
3,Charlie,Chennai
4,Diana,Pune


orders (large side):


order_id,customer_id,amount
101,1,250.0
102,2,80.5
103,1,120.75
104,3,300.0
105,2,45.25
106,4,99.99


In [0]:
# 1) BROADCAST JOIN (BroadcastHashJoin)
# The small side (customers) is collected on the driver and broadcast to every executor,
# so the join happens map-side with NO shuffle of either table.
# Spark does this automatically when a side is under
# spark.sql.autoBroadcastJoinThreshold (10 MB by default).
# Force it with F.broadcast(...) or the SQL hint /*+ BROADCAST(customers) */
from pyspark.sql import functions as F

broadcast_join = orders.join(F.broadcast(customers), "customer_id")

print("Plan -> look for BroadcastHashJoin and no Exchange (shuffle) before the join:")
broadcast_join.explain()

display(broadcast_join.orderBy("order_id"))

Plan -> look for BroadcastHashJoin and no Exchange (shuffle) before the join:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [customer_id#10158, order_id#10157, amount#10159, name#10140, city#10141]
         +- PhotonBroadcastHashJoin [customer_id#10158], [customer_id#10139], Inner, BuildRight, false, true, false
            :- PhotonRowToColumnar
            :  +- LocalTableScan [order_id#10157, customer_id#10158, amount#10159]
            +- PhotonShuffleExchangeSource false
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#6968]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [customer_id#10139, name#10140, city#10141]


== Photon Explanation ==
The query is fully supported by Photon.


customer_id,order_id,amount,name,city
1,101,250.0,Alice,Hyderabad
2,102,80.5,Bob,Bengaluru
1,103,120.75,Alice,Hyderabad
3,104,300.0,Charlie,Chennai
2,105,45.25,Bob,Bengaluru
4,106,99.99,Diana,Pune


In [0]:
# 2) SHUFFLE JOIN (ShuffledHashJoin)
# BOTH tables are repartitioned (shuffled) by the join key so matching keys land in
# the same partition, then an in-memory hash table is built from the smaller side.
# Forced here with the shuffle_hash hint.

shuffle_join = orders.hint("shuffle_hash").join(customers, "customer_id")

print("Plan -> look for ShuffledHashJoin; the Exchange steps are the shuffles:")
shuffle_join.explain()

display(shuffle_join.orderBy("order_id"))

Plan -> look for ShuffledHashJoin; the Exchange steps are the shuffles:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [customer_id#10158, order_id#10157, amount#10159, name#10140, city#10141]
         +- PhotonShuffledHashJoin [customer_id#10158], [customer_id#10139], Inner, BuildLeft
            :- PhotonShuffleExchangeSource false
            :  +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7195]
            :     +- PhotonShuffleExchangeSink hashpartitioning(customer_id#10158, 16)
            :        +- PhotonRowToColumnar
            :           +- LocalTableScan [order_id#10157, customer_id#10158, amount#10159]
            +- PhotonShuffleExchangeSource false
               +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7200]
                  +- PhotonShuffleExchangeSink hashpartitioning(customer_id#10139, 16)
                     +- PhotonRowToColumnar
                

customer_id,order_id,amount,name,city
1,101,250.0,Alice,Hyderabad
2,102,80.5,Bob,Bengaluru
1,103,120.75,Alice,Hyderabad
3,104,300.0,Charlie,Chennai
2,105,45.25,Bob,Bengaluru
4,106,99.99,Diana,Pune


In [0]:
# 3) SORT-MERGE JOIN (SortMergeJoin)
# Both sides are shuffled by the join key AND sorted, then merged with a zipper-style
# merge cursor. This is Spark's DEFAULT strategy when neither side is small enough
# to broadcast. Forced here with the merge hint.

merge_join = orders.hint("merge").join(customers, "customer_id")

print("Plan -> look for SortMergeJoin; note the Sort steps after each Exchange:")
merge_join.explain()

display(merge_join.orderBy("order_id"))

Plan -> look for SortMergeJoin; note the Sort steps after each Exchange:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   Project [customer_id#10158, order_id#10157, amount#10159, name#10140, city#10141]
   +- SortMergeJoin [customer_id#10158], [customer_id#10139], Inner
      :- ColumnarToRow
      :  +- PhotonResultStage
      :     +- PhotonSort [customer_id#10158 ASC NULLS FIRST]
      :        +- PhotonShuffleExchangeSource false
      :           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7463]
      :              +- PhotonShuffleExchangeSink hashpartitioning(customer_id#10158, 16)
      :                 +- PhotonRowToColumnar
      :                    +- LocalTableScan [order_id#10157, customer_id#10158, amount#10159]
      +- ColumnarToRow
         +- PhotonResultStage
            +- PhotonSort [customer_id#10139 ASC NULLS FIRST]
               +- PhotonShuffleExchangeSource false
                  +- PhotonShuffleMapStage ENSURE_REQUI

customer_id,order_id,amount,name,city
1,101,250.0,Alice,Hyderabad
2,102,80.5,Bob,Bengaluru
1,103,120.75,Alice,Hyderabad
3,104,300.0,Charlie,Chennai
2,105,45.25,Bob,Bengaluru
4,106,99.99,Diana,Pune


## When to use which

| Strategy | Shuffle cost | Sort cost | Memory | Best when |
| --- | --- | --- | --- | --- |
| Broadcast hash join | none | none | one side held in executor memory | one side is small (default < 10 MB) |
| Shuffle hash join | both sides shuffled | no sort | hash table per partition | one side medium-sized, fits in partition memory |
| Sort-merge join | both sides shuffled | both sides sorted | low | two large tables — Spark's default |

* Spark's `BROADCAST` / `SHUFFLE_HASH` / `MERGE` join hints force a strategy; without hints Spark chooses automatically based on table sizes.

In [0]:
# BONUS: MERGE INTO (Delta upsert) — in DML terms "merge" also means atomically
# INSERT new rows + UPDATE existing rows in a Delta table in one statement.

updates = spark.createDataFrame(
    [
        (1, "Alice", "Hyderabad-Updated"),  # existing customer -> UPDATE
        (5, "Eve",   "Delhi"),              # new customer     -> INSERT
    ],
    schema="customer_id INT, name STRING, city STRING",
)

try:
    ctx = spark.sql("SELECT current_catalog() AS cat, current_schema() AS sch").first()
    target_table = f"{ctx['cat']}.{ctx['sch']}.demo_merge_customers"

    customers.write.mode("overwrite").saveAsTable(target_table)
    updates.createOrReplaceTempView("updates")

    spark.sql(f"""
        MERGE INTO {target_table} AS t
        USING updates AS s
          ON t.customer_id = s.customer_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"Merged into {target_table}:")
    display(spark.table(target_table).orderBy("customer_id"))
except Exception as e:
    print("Skipped MERGE INTO demo — no permission to create a table in the current schema.")
    print(e)

Merged into workspace.default.demo_merge_customers:


customer_id,name,city
1,Alice,Hyderabad-Updated
2,Bob,Bengaluru
3,Charlie,Chennai
4,Diana,Pune
5,Eve,Delhi


## More examples

Three more hands-on demos:
1. **Auto-broadcast** — what Spark picks with no hints, and how `autoBroadcastJoinThreshold` flips it
2. **Data skew + salting** — the classic shuffle-join killer and its manual fix
3. **Advanced MERGE INTO** — conditional updates and deleting rows that vanished from the source

In [0]:
# 4) AUTO-BROADCAST IN ACTION (no hints)
# customers is tiny, so Spark broadcasts it automatically — even with NO hint
# the plan shows BroadcastHashJoin.

auto_join = orders.join(customers, "customer_id")

print("Default config -> BroadcastHashJoin (automatic):")
auto_join.explain()

# Now disable auto-broadcast: the SAME join falls back to SortMergeJoin.
try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

    auto_join2 = orders.join(customers, "customer_id")
    print("\nWith autoBroadcastJoinThreshold = -1 -> SortMergeJoin:")
    auto_join2.explain()

    # Reset the default (10 MB) so later cells behave normally.
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)
except Exception:
    # On Serverless compute this conf is read-only, so we cannot flip it here.
    # The rule it controls: with NO side under the 10 MB threshold, the same join
    # falls back to SortMergeJoin — exactly the plan shown by the merge-hint example above.
    print("spark.sql.autoBroadcastJoinThreshold cannot be changed on this compute —")
    print("without a small side, the same join falls back to SortMergeJoin (see the merge-hint example).")

Default config -> BroadcastHashJoin (automatic):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [customer_id#10158, order_id#10157, amount#10159, name#10140, city#10141]
         +- PhotonBroadcastHashJoin [customer_id#10158], [customer_id#10139], Inner, BuildLeft, false, true, false
            :- PhotonShuffleExchangeSource false
            :  +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#10895]
            :     +- PhotonShuffleExchangeSink SinglePartition
            :        +- PhotonRowToColumnar
            :           +- LocalTableScan [order_id#10157, customer_id#10158, amount#10159]
            +- PhotonRowToColumnar
               +- LocalTableScan [customer_id#10139, name#10140, city#10141]


== Photon Explanation ==
The query is fully supported by Photon.
spark.sql.autoBroadcastJoinThreshold cannot be changed on this compute —
without a small side, the same join falls 

In [0]:
# 5) DATA SKEW: the enemy of shuffle joins, and the SALTING fix
# Customer 1 owns 2000 of the 2010 orders. In a shuffle join, every row of
# customer 1 lands in ONE partition -> one long straggler task.
from pyspark.sql import functions as F

heavy = spark.range(2000).select(
    F.col("id").alias("order_id"),
    F.lit(1).cast("int").alias("customer_id"),
    F.round(F.rand(42) * 100, 2).alias("amount"),
)
light = spark.range(2000, 2010).select(
    F.col("id").alias("order_id"),
    (F.col("id") % 3 + 2).cast("int").alias("customer_id"),
    F.round(F.rand(7) * 100, 2).alias("amount"),
)
skewed_orders = heavy.unionByName(light)

print("Orders per customer — almost everything is customer 1:")
display(skewed_orders.groupBy("customer_id").count().orderBy("customer_id"))

# SALTING: spread the hot key over SALT random buckets on the fact side ...
SALT = 8
salted_orders = skewed_orders.withColumn("salt", (F.rand(seed=1) * SALT).cast("int"))

# ... and replicate each dimension row once per salt bucket so every bucket still matches
salted_customers = customers.withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(SALT - 1))))

salted_join = salted_orders.join(salted_customers, ["customer_id", "salt"])

# The salted join returns exactly the same rows as the plain join (2010 = every order matches):
print(f"plain join rows:  {skewed_orders.join(customers, 'customer_id').count()}")
print(f"salted join rows: {salted_join.count()}")

# NOTE: modern Spark's AQE (spark.sql.adaptive.skewJoin.enabled, on by default)
# also splits skewed sort-merge partitions automatically — salting is the manual fix
# for when AQE can't help (e.g. shuffled hash joins or extreme hot keys).

Orders per customer — almost everything is customer 1:


customer_id,count
1,2000
2,3
3,3
4,4


plain join rows:  2010
salted join rows: 2010


In [0]:
# 6) ADVANCED MERGE: conditional UPDATE + DELETE of vanished rows
# Keep the target table in exact sync with the source:
#   UPDATE only rows whose values actually changed,
#   INSERT new rows,
#   DELETE target rows that disappeared from the source (WHEN NOT MATCHED BY SOURCE).

fresh_customers = spark.createDataFrame(
    [
        (1, "Alice",   "Hyderabad-Updated"),  # unchanged  -> no-op
        (2, "Bob",     "Secunderabad"),       # changed    -> UPDATE
        (3, "Charlie", "Chennai"),            # unchanged  -> no-op
        (5, "Eve",     "Delhi"),              # unchanged  -> no-op
        (6, "Frank",   "Kolkata"),            # new        -> INSERT
        # customer 4 (Diana) is missing from the source -> DELETE from target
    ],
    schema="customer_id INT, name STRING, city STRING",
)
fresh_customers.createOrReplaceTempView("fresh_customers")

ctx = spark.sql("SELECT current_catalog() AS cat, current_schema() AS sch").first()
target_table = f"{ctx['cat']}.{ctx['sch']}.demo_merge_customers"

# Rebuild the target if the earlier MERGE demo never ran in this session:
if not spark.catalog.tableExists(target_table):
    customers.write.mode("overwrite").saveAsTable(target_table)

spark.sql(f"""
    MERGE INTO {target_table} AS t
    USING fresh_customers AS s
      ON t.customer_id = s.customer_id
    WHEN MATCHED AND t.city <> s.city THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")

print("Target after merge (Bob updated, Frank inserted, Diana deleted):")
display(spark.table(target_table).orderBy("customer_id"))

Target after merge (Bob updated, Frank inserted, Diana deleted):


customer_id,name,city
1,Alice,Hyderabad-Updated
2,Bob,Secunderabad
3,Charlie,Chennai
5,Eve,Delhi
6,Frank,Kolkata
